# Etapa 1 — EDA, preprocesamiento y detección de *data leakage*
**Autores:** Edwin Yunis y Jairo Serrano  
**Machine Learning · Dr. Lihki Rubio**

Demostración del error de ajustar el preprocesamiento con registros del conjunto de prueba. Se trabaja exclusivamente con el dataset original, sin crear observaciones ni variables.

In [ ]:
import sys
from pathlib import Path
# Compatible con ejecución tanto desde la raíz como desde notebooks/.
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_data, describe_data, prepare_features
from src.leakage import compare_leakage
import pandas as pd
import matplotlib.pyplot as plt

## Datos auténticos
Se obtiene exclusivamente el dataset [Heart Failure Prediction de Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction). `HeartDisease` es su variable objetivo; el ejemplo del profesor usa `target` para ilustrar un flujo genérico.

In [ ]:
df = load_data(ROOT / "data" / "heart.csv")
print(pd.Series(describe_data(df)).to_string())
display(df.head())
display(df.isna().sum().rename("faltantes explícitos").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df["HeartDisease"].value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set_title("Clases originales")
axes[0].set_xlabel("HeartDisease (0/1)")
axes[0].tick_params(axis="x",rotation=0)
for label, subset in df.groupby("HeartDisease"):
    axes[1].hist(subset["Age"], bins=16, alpha=0.55, label=f"Clase {label}")
axes[1].set_title("Edad por clase")
axes[1].set_xlabel("Edad (años)")
axes[1].legend()
plt.tight_layout()
plt.show()

## Flujo incorrecto frente a flujo válido
El flujo incorrecto estima imputación, escalado y codificación usando el conjunto completo antes de separar test, tal como el ejemplo docente advierte que no debe hacerse. El flujo correcto aprende las transformaciones dentro de `Pipeline` únicamente a partir del entrenamiento. No se crea `leaky_feature`, porque en esta entrega se ha solicitado utilizar únicamente los datos originales.

In [ ]:
resultado_fuga = compare_leakage(df)
display(pd.Series(resultado_fuga).rename("AUC").to_frame())

## Interpretación y decisión
La fuga de preprocesamiento no necesariamente mejora AUC: cualquier diferencia depende de los registros observados. La regla metodológica es no ajustar ninguna transformación usando el conjunto de prueba. Los cinco modelos finales usan siempre la modalidad válida y los 11 predictores originales.